# Visualização de dados do mercado financeiro

Autor: Felipe Pasinato Rossoni


O dataset escolhido para a visualização de dados foi um conjunto de indicadores do mercado financeiro, como o índice Ibovespa, o índice S&P500, o valor do dólar comercial (USD/BRL) e algumas ações da B3 (Bolsa do Brasil).

Alguns dos dados interessantes que podem ser analisados dentro do dataset serão: rentabilidades de índices e ações, volume mensal de negociação e comparação de índices com o cenário macroecônomico.

Para o funcionamento correto do notebook, é necessário ir na aba de arquivos e fazer o upload de todos os arquivos .csv contidos na pasta em que o notebook foi carregado.

## 1. Noções básicas
Considerando que o assunto é sobre o mercado financeiro, aqui estão algumas explicações sobre alguns termos que vão aparecer neste notebook:

Ibovespa (IBOV): principal índice da bolsa brasileira, usado como referência do mercado acionário no Brasil.

S&P500: principal índice de ações dos Estados Unidos, formado por 500 grandes empresas.

Rentabilidade: variação percentual do valor de um ativo em um período.

Volatilidade: medida da intensidade das oscilações de preço ou retorno de um ativo.

Ticker: código de negociação do ativo na bolsa, formado por 4 letras e 1 número.

Cotação: preço de negociação de um ativo em um determinado momento.

Papéis: ações ou ativos negociados no mercado.

Volume de negociação: quantidade de ações negociadas em um período.

Dividendos: parte do lucro distribuída aos acionistas.

Desdobramentos: divisão das ações em mais unidades, sem alterar o valor total investido.

Taxa de juros: custo do dinheiro ao longo do tempo, usada como referência para investimentos.

CDI: taxa de referência das operações entre bancos e um dos principais indicadores da renda fixa no Brasil, acompanha a taxa de juros.

Aversão ao risco: comportamento de preferência por ativos mais seguros em momentos de incerteza.

Apetite ao risco: cenário em que investidores assumem mais risco buscando maiores retornos.

Pregão: período diário em que ocorrem as negociações na bolsa de valores, onde ativos são comprados e vendidos pelos investidores.

Commodities: produtos básicos negociados em escala global, como petróleo e minério de ferro.

## 2. Carregar bibliotecas



In [ ]:
!pip install pandas
!pip install altair
!pip install numpy
import pandas as pd
import altair as alt
import numpy as np

## 3. Preparação de dados
Nesta etapa, os dados são formatados para posteriormente serem utilizados nos gráficos. Os dados de índices e ações foram previamente formatados manualmente no quesito de datas e remoção de informações como dividendos e desdobramentos de ações. Já o índice do CDI precisa ser formatado com mais cuidado porque está em um formato diferente do resto dos ativos.
O índice do CDI foi obtido da página do Sistema Gerenciador de Séries Temporais do Banco Central do Brasil, enquanto os outros índices e ações foram obtidos do Yahoo Finance.
Também é calculada a rentabilidade acumulada dos ativos e é feita uma classificação por setores.


In [ ]:
alt.data_transformers.disable_max_rows()
DATA_INICIO = '2000-01-03'

def carregar_dados_completo():

    #calculo cdi mensal
    df_cdi = pd.read_csv('CDI.csv', sep=';')
    df_cdi.columns = ['Data', 'Taxa']
    df_cdi['Data'] = pd.to_datetime(df_cdi['Data'], format='%m/%Y', errors='coerce')
    df_cdi = df_cdi.dropna(subset=['Data'])
    df_cdi['Taxa'] = pd.to_numeric(df_cdi['Taxa'], errors='coerce')
    df_cdi = df_cdi.sort_values('Data')
    df_cdi['Rentabilidade_%'] = ((1 + df_cdi['Taxa'] / 100).cumprod() - 1) * 100
    df_cdi['Ativo'] = 'CDI'
    df_cdi['Preco'] = 0.0
    df_cdi['Volume'] = 0.0

    # ações e indices
    ativos = ['IBOV', 'SP500', 'USDBRL', 'PETR4', 'VALE3', 'ABEV3', 'ITUB4', 'BBAS3']
    setores = {
        'IBOV': 'Índice', 'SP500': 'Índice', 'USDBRL': 'Moeda',
        'PETR4': 'Commodities', 'VALE3': 'Commodities',
        'ABEV3': 'Consumo', 'ITUB4': 'Financeiro', 'BBAS3': 'Financeiro'
    }

    lista_dfs = []
    for a in ativos:
        temp = pd.read_csv(f'{a}.csv', thousands=',', na_values='-', parse_dates=['Date'])
        temp = temp.sort_values('Date').query("Date >= @DATA_INICIO")

        inicio_preco = temp['Adj Close'].iloc[0]
        temp['Rentabilidade_%'] = ((temp['Adj Close'] / inicio_preco) - 1) * 100
        temp['Retorno_Diario'] = temp['Adj Close'].pct_change()
        temp['Ativo'] = a
        temp['Setor'] = setores[a]
        temp = temp.rename(columns={'Date': 'Data', 'Adj Close': 'Preco'})
        lista_dfs.append(temp[['Data', 'Rentabilidade_%', 'Retorno_Diario', 'Ativo', 'Preco', 'Volume', 'Setor']])

    df_mkt = pd.concat(lista_dfs)
    return df_mkt, df_cdi

df_mercado, df_cdi_final = carregar_dados_completo()

## 4. Gráfico de rentabilidade acumulada
Este gráfico compara a rentabilidade acumulada dos índices Ibovespa e S&P500, do CDI e da taxa de câmbio do dólar.
É possível ver que, no período analisado (desde o ínicio do ano 2000), o CDI superou com folga os demais investimentos domésticos. Isso acontece porque o Brasil é um país com taxas de juros historicamente elevadas, proporcionando um retorno maior aos investimentos atrelados à renda fixa. Enquanto isso, a comparação entre IBOV e S&P500 deve ser feita com cautela, pois os índices estão em moedas diferentes (real e dólar), o que pode distorcer a análise de rentabilidade relativa.

In [ ]:
indices_plot = pd.concat([df_mercado[df_mercado['Ativo'].isin(['IBOV', 'SP500', 'USDBRL'])], df_cdi_final])

chart_profitability = alt.Chart(indices_plot).mark_line().encode(
    x=alt.X('Data:T', title='Ano'),
    y=alt.Y('Rentabilidade_%:Q', title='Rentabilidade Acumulada (%)'),
    color=alt.Color('Ativo:N', scale=alt.Scale(scheme='set1')),
    tooltip=['Data:T', 'Ativo:N', alt.Tooltip('Rentabilidade_%:Q', format='.2f')]
).properties(
    width=900, height=450,
    title='Rentabilidade Acumulada'
).interactive()
chart_profitability

## 5. Comparação de 5 ações da B3
Foram escolhidas para análise de rentabilidade 5 grandes ações que compoem a carteira do índice Ibovespa. Assim, é possível ver a variação de rentabilidade ao longos dos anos dessas empresas, considerando sua rentabilidade ajustada, a qual consiste no reinvestimento de todos os dividendos recebidos. Abaixo está uma breve contextualização de cada empresa, seu ticker, seu setor e sua participação percentual na carteira do índice.

Essas ações são:

Petrobras (PETR4): Empresa estatal brasileira de petróleo e gás, com atuação em exploração, produção e refino. Setor: Energia (Petróleo, Gás e Biocombustíveis). Participação: 8,274%

Banco do Brasil (BBAS3): Instituição financeira estatal com forte atuação em crédito, varejo bancário e agronegócio. Setor: Financeiro (Bancos). Participação: 2,445%

Itaú (ITUB4): Maior banco privado da América Latina, com atuação em diversos segmentos financeiros. Setor: Financeiro (Bancos). Participação: 7,912%

Vale (VALE3): Uma das maiores mineradoras do mundo, focada na produção de minério de ferro e metais. Setor: Materiais Básicos (Mineração). Participação: 11,063%

Ambev (ABEV3): Empresa líder no setor de bebidas, com foco em cervejas e refrigerantes. Setor: Consumo (Bebidas). Participação: 2,395%



In [ ]:
acoes_list = ['PETR4', 'VALE3', 'ABEV3', 'ITUB4', 'BBAS3']
df_acoes = df_mercado[df_mercado['Ativo'].isin(acoes_list)]

chart_stocks = alt.Chart(df_acoes).mark_line().encode(
    x='Data:T',
    y=alt.Y('Rentabilidade_%:Q', title='Rentabilidade (%)'),
    color='Ativo:N',
    tooltip=['Data:T', 'Ativo:N', 'Preco:Q', alt.Tooltip('Rentabilidade_%:Q', format='.2f')]
).properties(
    width=900,
    height=500,
    title='Performance Comparativa de Ações da B3'
).interactive()

chart_stocks

## 6. Comparação de 5 ações da B3 junto com outros índices
Aqui foi feita uma comparação da rentabilidade das ações juntamente com os outros índices do mercado, para se ter uma melhor visão geral dos retornos ao longo dos anos. Assim podemos visualizar a diferença entre investir em índices e investir individualmente em certas empresas.

In [ ]:
ranking_mercado = df_mercado.groupby('Ativo').last().reset_index()
ranking_cdi = df_cdi_final.iloc[[-1]].copy()
df_ranking = pd.concat([ranking_mercado[['Ativo', 'Rentabilidade_%']], ranking_cdi[['Ativo', 'Rentabilidade_%']]])

chart_stocks_bar = alt.Chart(df_ranking).mark_bar().encode(
    x=alt.X('Rentabilidade_%:Q',
            title='Retorno Total Acumulado (%)',
            axis=alt.Axis(format='~s', tickCount=10)), # 's' simplifica 6000 para 6k
    y=alt.Y('Ativo:N', sort='-x', title='Ativo'),
    color=alt.Color('Rentabilidade_%:Q', scale=alt.Scale(scheme='greens'), legend=None),
    tooltip=[alt.Tooltip('Ativo:N'), alt.Tooltip('Rentabilidade_%:Q', format=',.2f')]
).properties(
    width=900, height=400,
    title='Ranking de Rentabilidade (Retorno Ajustado)'
)

chart_stocks_bar

## 7. Avaliação da rentabilidade x risco dos ativos
Neste gráfico podemos visualizar o quanto um ativo é arriscado (maior volatilidade) comparado com a sua rentabilidade. O eixo y indica rentabilidade, ou seja, cada vez maior o valor de y, maior foi o retorno de investir no ativo, enquanto o eixo x indica a sua volatilidade, ou seja, a sua cotação oscilou mais significativamente durante o ano.

Com esse gráfico, podemos ver que ações e índices mais a direita do gráfico são mais arriscadas, enquanto ações e índices mais para cima do gráfico são mais rentáveis para o investidor.

In [ ]:
df_plot = df_mercado[df_mercado['Ativo'] != 'CDI']

stats = df_plot.groupby('Ativo').agg(
    Volatilidade_Anual=('Retorno_Diario', lambda x: x.std() * np.sqrt(252)),
    Retorno_Anual=('Retorno_Diario', lambda x: (1 + x.dropna()).prod() ** (252 / len(x.dropna())) - 1)
).reset_index()

chart_risk_profitability = alt.Chart(stats).mark_circle(size=180).encode(
    x=alt.X('Volatilidade_Anual:Q', title='Volatilidade Anualizada (Risco)', axis=alt.Axis(format='.0%')),
    y=alt.Y('Retorno_Anual:Q', title='Retorno Anualizado', axis=alt.Axis(format='.0%')),
    color=alt.Color('Ativo:N', scale=alt.Scale(scheme='category10'), title='Ativo'),
    tooltip=[
        'Ativo',
        alt.Tooltip('Retorno_Anual:Q', format='.2%'),
        alt.Tooltip('Volatilidade_Anual:Q', format='.2%')
    ]
).properties(
    width=800,
    height=450,
    title='Rentabilidade x Risco por Ativo'
).interactive()

chart_risk_profitability

## 8. Volume de negociação de ações
Neste gráfico é mostrado o volume de negociação das ações do Itaú, Petrobras e Vale desde 2018 até 2026. Esse volume, organizado por meses, indica o acumulado de quantos papéis foram negociados durante o mês, ou seja, quantas ações foram negociadas nos pregões durante esse período de tempo.

In [ ]:
df_recente = df_mercado[(df_mercado['Data'] >= '2018-01-01') & (df_mercado['Ativo'].isin(['PETR4', 'VALE3', 'ITUB4']))]

chart_vol = alt.Chart(df_recente).mark_bar(opacity=1).encode(
    x=alt.X('yearmonth(Data):T', title='Tempo (Mês/Ano)'),
    y=alt.Y('sum(Volume):Q',
            title='Volume Financeiro Total',
            axis=alt.Axis(format='~s', labelExpr="replace(datum.label, 'G', 'B')")), # Troca G por B
    color=alt.Color('Ativo:N', scale=alt.Scale(scheme='set1')),
    tooltip=['Ativo', alt.Tooltip('sum(Volume):Q', format=',.0f')]
).properties(
    width=900, height=400,
    title='Volume de Negociação Mensal'
)

chart_vol

### 9. Apetite ao risco comparado com o cenário macroeconômico
Nesta seção foi feita uma comparação entre o índice Ibovespa e a taxa de câmbio do dólar, os quais são indicadores muito afetados pelo cenário macroeconômico.

É possível observar três momentos de forte estresse no mercado. A Crise do Subprime (2007–2008) foi uma crise financeira global que começou no mercado imobiliário dos Estados Unidos e gerou forte aversão ao risco, derrubando bolsas no mundo todo. Nesse período, a bolsa brasileira também sofreu quedas, enquanto o dólar ganhou força como ativo de proteção.

A segunda ocorrência foi a crise econômica brasileira de 2014–2016, marcada por recessão, instabilidade política e piora da confiança dos investidores. Nesse cenário, o Ibovespa teve desempenho fraco e o dólar subiu, refletindo a fuga de capital e a desvalorização do real.

Por fim, a pandemia de COVID-19 em 2020 provocou uma forte incerteza global no início da crise, com queda brusca nas bolsas e valorização do dólar. Em seguida, com a retomada gradual dos mercados, o Ibovespa se recuperou gradualmente, mas o câmbio continuou sensível às incertezas econômicas.

In [ ]:
eventos = pd.DataFrame([
    {'Data': '2008-09-15', 'Evento': 'Crise Subprime'},
    {'Data': '2014-04-30', 'Evento': 'Crise político-econômica'},
    {'Data': '2020-03-11', 'Evento': 'Pandemia COVID-19'}
])
eventos['Data'] = pd.to_datetime(eventos['Data'])

base_lines = alt.Chart(df_mercado[df_mercado['Ativo'].isin(['IBOV', 'USDBRL'])]).mark_line().encode(
    x='Data:T',
    y='Rentabilidade_%:Q',
    color='Ativo:N'
)

regras = alt.Chart(eventos).mark_rule(color='red', strokeDash=[3,3]).encode(x='Data:T')

labels = alt.Chart(eventos).mark_text(
    align='left', dx=5, dy=-200, angle=0, color='red', fontWeight='bold'
).encode(x='Data:T', text='Evento')

(base_lines + regras + labels).properties(
    width=900, height=500,
    title='Comportamento do Mercado em Grandes Crises'
).interactive()

## 10. Avaliação de ações por setor
Considerando os setores das ações, nota-se que o setor de Commodities (VALE3 e PETR4) apresenta maior volatilidade ao longo do tempo, com ciclos mais acentuados de alta e queda, refletindo sua forte dependência de fatores externos como preços internacionais e demanda global.

Em contrapartida, o setor Financeiro (ITUB4 e BBAS3) demonstra uma trajetória mais estável e consistente, com crescimento gradual e menor oscilação relativa. No longo prazo, ambos os setores apresentam valorização relevante, mas o setor de commodities se destaca por picos mais elevados de rentabilidade, enquanto o financeiro tende a oferecer um comportamento mais previsível.


In [ ]:
df_setores = df_mercado[df_mercado['Setor'].isin(['Financeiro', 'Commodities'])]

chart_setor = alt.Chart(df_setores).mark_line().encode(
    x='Data:T',
    y=alt.Y('Rentabilidade_%:Q', title='Rentabilidade (%)'),
    color='Ativo:N',
    tooltip=['Ativo', 'Rentabilidade_%:Q']
).properties(
    width=400, height=300
).facet(
    column='Setor:N',
    title='Performance por Setor de Atuação'
)

chart_setor